# 6장. AI 예측 모델로 후크의 응력/변위를 예측하고 평가하기

## 개요

**목표:** 5장 선택 모델의 시험 케이스 예측 → ANSYS 정답과 오차 비교 → VTU/평가 기록 저장 및 7장 연결

| 순서 | 직접 하는 일 |
|---|---|
| 1. 모델 불러오기 | 5장 연결 정보 읽기, Transolver/선택 에폭/GPU 확인 |
| 2. 시험 입력 준비 | 한 케이스의 전체 좌표/두께/하중/노드 순서 확인 |
| 3. 전체 노드 추론 | 입력 정규화 → 예측 → MPa/mm로 역정규화 |
| 4. 그림으로 비교 | 정답/예측/오차를 같은 실제 메시에서 관찰 |
| 5. 지표로 평가 | MAE/최대값/위치/RMSE 확인 |
| 6. 두 케이스 평가와 저장 | 같은 모델의 두 케이스 결과 비교 → 저장 → 다시 불러와 확인 |
| 7. 결과 정리 | 적용 범위와 7장에서 사용할 파일 기록 |


## 1. 학습한 모델 불러오기


### 1.1 평가할 모델과 학습 결과 확인


### 06장 / 셀 03 / Notebook


In [ ]:
from pathlib import Path
from datetime import datetime
import json, sys, subprocess
import numpy as np
import torch
from IPython.display import display, Image

working_dir = Path.cwd().resolve()
education_dir = next((p for base in [working_dir, *working_dir.parents]
                      for p in [base, base/'education']
                      if (p/'course_utils/dataset.py').is_file()), None)
if education_dir is None:
    raise FileNotFoundError('교육 폴더 또는 그 하위 폴더에서 실행하세요.')
if str(education_dir) not in sys.path:
    sys.path.insert(0, str(education_dir))
from course_utils.training import seed_training
from course_utils.inference import load_selected_model, predict_fields
from course_utils.transolver_evaluation import (
    evaluate_case, print_metrics, print_peaks, plot_case, save_evaluation)
seed_training(42)
if not torch.cuda.is_available():
    raise RuntimeError('1장의 GPU/CUDA용 PyTorch/커널을 확인하세요.')
device = torch.device('cuda')
print('교육 폴더:', education_dir)
print('GPU:', torch.cuda.get_device_name(device))


### 06장 / 셀 04 / Notebook


In [ ]:
from course_utils.training_handoff import resolve_training_handoff
# 기본값은 5장에서 연결한 본인의 모델입니다. 참고 모델로 진행할 때만 True로 바꿉니다.
use_reference_model = False
if use_reference_model:
    training_run = education_dir/'runs/training/student_transolver_1000_20261015_162702_786921'
    print('모델 출처: 제작 참고 모델(직접 학습한 결과와 구분)')
else:
    training_run = resolve_training_handoff(education_dir)
    print('모델 출처: 5장 9.3절에서 연결한 기본 학습 B')
model, normalizer, cases, split, fixed, context = load_selected_model(
    education_dir, training_dir=training_run, device=device)
assert getattr(model, 'requires_full_point_context', False)
frozen_weights = {k:v.detach().clone() for k,v in model.state_dict().items()}
print('계산 모델:', type(model.transolver).__module__+'.'+type(model.transolver).__name__)
print('학습 폴더:', training_run)
print('검증으로 선택한 에폭:', context['selected_epoch'])
print('모델 SHA256:', context['checkpoint_sha256'])
print('시험 케이스:', split['test'])
print('단위:', normalizer.stats['units'])
print('조건 순서:', normalizer.stats['condition_order'])


## 2. 시험 케이스의 입력 준비


### 2.1 같은 노드의 위치/조건/정답 연결


### 06장 / 셀 07 / Notebook


In [ ]:
case_id = split['test'][0]
case = cases[case_id]
print('시험 케이스:', case_id, '| 전체 노드:', len(case.coords))
print('좌표 / 조건:', case.coords.shape, case.globals.shape)
print('두께(mm) / 총하중(N):', case.globals.tolist())
print('정답 응력 / 변위:', case.stress.shape, case.displacement.shape)
print('원본 노드 ID 앞 5개:', case.node_ids[:5].tolist())


## 3. 전체 노드의 응력/변위를 예측하고 원래 단위로 되돌리기


### 3.2 정규화 → 추론 → 역정규화


### 06장 / 셀 09 / Notebook


In [ ]:
# 1. 한 케이스의 전체 입력을 정규화해 GPU로 전달합니다.
coords = torch.from_numpy(normalizer.normalize('coords', case.coords).astype(np.float32)).unsqueeze(0).to(device)
conditions = torch.from_numpy(normalizer.normalize('globals', case.globals).astype(np.float32)).unsqueeze(0).to(device)
model.eval()
# 2. 전체 노드를 한 번에 예측합니다.
with torch.no_grad():
    normalized_output = model((coords, conditions))
# 3. 같은 학습 통계로 MPa/mm를 복원합니다.
prediction = {field: normalizer.denormalize(field, value).squeeze(0).cpu().numpy().astype(np.float64)
              for field, value in normalized_output.items()}
assert prediction['stress'].shape == case.stress.shape
assert prediction['displacement'].shape == case.displacement.shape
assert all(np.isfinite(values).all() for values in prediction.values())
print('모델 입력 좌표/조건:', tuple(coords.shape), tuple(conditions.shape))
print('정규화한 출력:', {k:tuple(v.shape) for k,v in normalized_output.items()})
print('역정규화한 예측:', {k:v.shape for k,v in prediction.items()})
print('응력 MPa 앞 5개:', np.round(prediction['stress'][:5,0], 4))
print('변위 mm 앞 2개:\n', np.round(prediction['displacement'][:2], 6))
print('음수 응력 노드:', int((prediction['stress'] < 0).sum()))


## 4. ANSYS 정답/예측/오차를 그림으로 비교


### 06장 / 셀 12 / Notebook


In [ ]:
evaluation_dir = education_dir/'runs/evaluation'/('student_transolver_'+datetime.now().strftime('%Y%m%d_%H%M%S_%f'))
evaluation_dir.mkdir(parents=True, exist_ok=False)
figure_path = plot_case(education_dir, case, prediction, evaluation_dir/f'{case_id}_comparison.png')
display(Image(filename=str(figure_path)))
print('그림:', figure_path)


## 5. 숫자로 예측 정확도 평가


### 5.1 MAE와 공식 RMSE의 의미


### 06장 / 셀 14 / Notebook


In [ ]:
from physicsnemo.metrics.general.mse import rmse

# 역정규화한 응력 예측/정답을 같은 GPU에서 비교합니다.
stress_pred = torch.tensor(prediction['stress'], dtype=torch.float64, device=device)
stress_target = torch.tensor(case.stress, dtype=torch.float64, device=device)
stress_rmse = rmse(stress_pred, stress_target)
print('RMSE 계산 함수:', rmse.__module__+'.'+rmse.__name__)
print('응력 MAE(MPa):', torch.mean(torch.abs(stress_pred-stress_target)).item())
print('응력 RMSE(MPa):', stress_rmse.item())


### 5.2 변위의 크기와 방향을 함께 평가하기


### 06장 / 셀 16 / Notebook


In [ ]:
# 평가 합산에는 float64, 앞의 모델 추론에는 float32를 사용합니다.
# 이 변환이 이미 계산된 예측의 정밀도를 높이는 것은 아닙니다.
# 1. 변위 예측/정답도 같은 GPU에서 비교합니다.
disp_pred = torch.tensor(prediction['displacement'], dtype=torch.float64, device=device)
disp_target = torch.tensor(case.displacement, dtype=torch.float64, device=device)
# 2. 공식 함수로 성분별/벡터 RMSE를 계산합니다.
component_rmse = rmse(disp_pred, disp_target, dim=0)
vector_difference = torch.linalg.vector_norm(disp_pred-disp_target, dim=1)
vector_rmse = rmse(vector_difference, torch.zeros_like(vector_difference))

# 3. 공통 평가도 같은 공식 RMSE를 사용합니다. MAE/최대값은 별도 계산합니다.
full_metrics, full_peaks = evaluate_case(case, prediction)
np.testing.assert_allclose(stress_rmse.item(), full_metrics['stress_rmse_mpa'], rtol=1e-12)
np.testing.assert_allclose(component_rmse.cpu().numpy(),
    [full_metrics[f'displacement_{a}_rmse_mm'] for a in 'xyz'], rtol=1e-12)
np.testing.assert_allclose(vector_rmse.item(), full_metrics['displacement_vector_rmse_mm'], rtol=1e-12)
print('평가 장치/정밀도:', stress_pred.device, stress_pred.dtype)
print_metrics(full_metrics)
# 수식으로 직접 계산한 NumPy 기준과도 일치하는지 확인합니다.
manual_rmse = float(np.sqrt(np.mean((prediction['stress']-case.stress)**2)))
np.testing.assert_allclose(full_metrics['stress_rmse_mpa'], manual_rmse, rtol=1e-12)


### 5.3 최대값과 최대 위치는 따로 비교


### 06장 / 셀 18 / Notebook


In [ ]:
print_peaks(full_metrics, full_peaks)


## 6. 두 시험 케이스 평가와 결과 저장


### 6.1 같은 모델로 두 번째 케이스 평가


### 06장 / 셀 20 / Notebook


In [ ]:
# 공통 함수와 3절의 직접 계산이 일치하는지 확인합니다.
common_first = predict_fields(model, normalizer, case.coords, case.globals)
for field, tolerance in [('stress',1e-4),('displacement',1e-6)]:
    np.testing.assert_allclose(common_first[field], prediction[field], rtol=0, atol=tolerance)
predictions = {case_id: prediction}
second_id = split['test'][1]
second_case = cases[second_id]
predictions[second_id] = predict_fields(model, normalizer, second_case.coords, second_case.globals)
second_metrics, second_peaks = evaluate_case(second_case, predictions[second_id])
print_metrics(second_metrics)
second_figure = plot_case(education_dir, second_case, predictions[second_id], evaluation_dir/f'{second_id}_comparison.png')
display(Image(filename=str(second_figure)))
print_peaks(second_metrics, second_peaks)


### 6.2 두 시험 케이스의 결과 비교


### 06장 / 셀 22 / Notebook


In [ ]:
comparison_rows = [full_metrics, second_metrics]
print('전체 노드 지표 | '+ ' | '.join(row['case_id'] for row in comparison_rows))
for label, key in [
    ('응력 MAE(MPa)', 'stress_mae_mpa'),
    ('응력 RMSE(MPa)', 'stress_rmse_mpa'),
    ('변위 크기 MAE(mm)', 'displacement_magnitude_mae_mm'),
    ('변위 벡터 RMSE(mm)', 'displacement_vector_rmse_mm'),
    ('정답 최대응력(MPa)', 'fea_max_stress_mpa'),
    ('예측 최대응력(MPa)', 'ai_max_stress_mpa'),
    ('최대응력 값의 절대차(MPa)', 'max_stress_abs_error_mpa'),
    ('최대응력 위치 사이 거리(mm)', 'max_stress_location_distance_mm'),
    ('정답 최대 변위 크기(mm)', 'fea_max_displacement_mm'),
    ('예측 최대 변위 크기(mm)', 'ai_max_displacement_mm'),
    ('최대 변위 크기의 절대차(mm)', 'max_displacement_abs_error_mm'),
    ('최대 변위 위치 사이 거리(mm)', 'max_displacement_location_distance_mm'),
    ('음수 응력 노드 수', 'negative_stress_count')
]:
    print(label+' | '+' | '.join(f'{row[key]:.6g}' for row in comparison_rows))


### 6.3 이번 평가 파일 저장


### 06장 / 셀 25 / Notebook


In [ ]:
evaluation_report = save_evaluation(education_dir, evaluation_dir, training_run,
    context, cases, split, fixed, predictions, device)
print('평가 폴더:', evaluation_dir)
print('지표 저장 행 수:', sum(len(item['metrics']) for item in evaluation_report['cases'].values()))
print('저장 파일: test_metrics.csv, evaluation_report.json, evaluation_manifest.json')
for cid, item in evaluation_report['cases'].items():
    print('예측 VTU:', evaluation_dir/item['prediction_vtu'])


### 6.4 저장한 모델을 다시 불러와 결과 확인하기


### 06장 / 셀 27 / Notebook


In [ ]:
command = [sys.executable, str(education_dir/'preparation/evaluate_transolver.py'),
           '--audit', '--output-dir', str(evaluation_dir), '--device', str(device)]
completed = subprocess.run(command, capture_output=True, text=True, encoding='utf-8', errors='replace')
if completed.returncode:
    raise RuntimeError(completed.stdout+'\n'+completed.stderr)
print(completed.stdout)
reload_check = json.loads((evaluation_dir/'reload_check.json').read_text(encoding='utf-8'))
assert reload_check['passed'] and reload_check['model_sha256'] == context['checkpoint_sha256']
assert all(torch.equal(frozen_weights[k],v) for k,v in model.state_dict().items())
assert all(p.grad is None for p in model.parameters())
print('별도 프로세스 재로딩과 현재 모델의 가중치 유지 확인 완료')
for cid, check in reload_check['cases'].items():
    print(cid, '| 재로딩 최대차:', check['maximum_reload_difference'])


## 7. 결과 정리와 7장 준비


### 7.2 다음 장에서 사용할 자료


### 06장 / 셀 30 / Notebook


In [ ]:
from course_utils.lesson_handoff import save_evaluation_handoff
handoff_path = save_evaluation_handoff(education_dir, evaluation_dir)
print('7장 자동 연결 정보:', handoff_path)
print('7장에 전달할 평가 폴더:', evaluation_dir.resolve())
print('평가한 선택 에폭:', context['selected_epoch'])
print('평가한 모델 SHA256:', context['checkpoint_sha256'])
for cid in split['test']:
    print(cid, ':', evaluation_dir/evaluation_report['cases'][cid]['prediction_vtu'])
print('시험 평가 후 모델 재선택/가중치 조정 없음')


### 7.4 선택 보충: 전체 평가와 고정 4,096개 노드 평가


### 06장 / 셀 33 / Notebook


In [ ]:
sample_metrics, sample_peaks = evaluate_case(case, prediction, fixed[case_id])
print('범위 | 노드 수 | 응력 MAE(MPa) | 정답 최대응력(MPa) | 예측 최대응력(MPa)')
for row in [full_metrics, sample_metrics]:
    print(row['scope'], row['n_points'], f"{row['stress_mae_mpa']:.6f}",
          f"{row['fea_max_stress_mpa']:.6f}", f"{row['ai_max_stress_mpa']:.6f}")
